In [ ]:
# ============================================================
# Cellule 1 — Imports et config
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from PIL import Image
import timm
import torch
import torch.nn as nn

from src.config import IMG_DIR

MODEL_NAME  = "vit_base_patch16_dinov3.lvd1689m"
CKPT        = "../dino_trainval_final.pt"
HIDDEN_SIZE = 128

# 256x256 → 16x16 = 256 patches de 16 px
GRID     = 16    # 16×16 patches
PATCH_PX = 16    # 16 px par patch → 256×256

DEVICE = torch.device("cuda" if torch.cuda.is_available()
                       else "mps" if torch.backends.mps.is_available()
                       else "cpu")
print("Device :", DEVICE)

In [ ]:
# ============================================================
# Cellule 2 — Définition du modèle (identique à train_clean_trainval.py)
# ============================================================
# Différence avec le notebook LoRA d'origine :
#   - pas de LoRA, le backbone entier a été fine-tuné
#   - num_prefix_tokens lu dynamiquement sur le backbone (5 = CLS + 4 register)
#   - forward renvoie (ratio, probs) pour pouvoir visualiser les cartes par patch

class PatchHead(nn.Module):
    def __init__(self, in_dim=768, hidden=HIDDEN_SIZE):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 3),   # fond / visible / occludé
        )

    def forward(self, patch_tokens):
        logits = self.net(patch_tokens)          # (B, 196, 3)
        probs  = torch.softmax(logits, dim=-1)
        p_visible  = probs[:, :, 1]
        p_occluded = probs[:, :, 2]
        ratio = p_occluded.sum(dim=1) / (
            p_visible.sum(dim=1) + p_occluded.sum(dim=1) + 1e-8
        )
        return ratio.unsqueeze(1), probs         # on renvoie aussi les probs !


class PatchOcclusionModel(nn.Module):
    """Backbone DINOv3 + tête patch. self.model = backbone (nom stable)."""
    def __init__(self, backbone):
        super().__init__()
        self.model = backbone
        embed_dim  = getattr(backbone, "embed_dim", 768)
        self.num_prefix = getattr(backbone, "num_prefix_tokens", 5)
        self.head = PatchHead(in_dim=embed_dim)

    def forward(self, x):
        features     = self.model.forward_features(x)        # (B, N, 768)
        patch_tokens = features[:, self.num_prefix:, :]       # (B, 196, 768)
        return self.head(patch_tokens)

In [ ]:
# ============================================================
# Cellule 3 — Construction et chargement du checkpoint
# ============================================================
# Le checkpoint 2026-06-11_23-02-27_full.pt contient TOUS les poids
# (backbone fine-tuné + tête), donc pretrained=False : on charge tout
# depuis le state_dict, pas depuis les poids DINOv3 d'origine.

backbone = timm.create_model(MODEL_NAME, pretrained=False, num_classes=0)
model    = PatchOcclusionModel(backbone)
print("num_prefix_tokens :", model.num_prefix)

state_dict = torch.load(CKPT, map_location="cpu")
missing, unexpected = model.load_state_dict(state_dict, strict=False)
print("Manquantes :", len(missing), "| Inattendues :", len(unexpected))
if missing:
    print("  -> manquantes :", missing[:10], "..." if len(missing) > 10 else "")
if unexpected:
    print("  -> inattendues :", unexpected[:10], "..." if len(unexpected) > 10 else "")

model = model.to(DEVICE).eval()

# Transform TIMM (identique à l'entraînement, is_training=False -> déterministe)
data_config = timm.data.resolve_model_data_config(
    timm.create_model(MODEL_NAME, pretrained=False)
)
transform = timm.data.create_transform(**data_config, is_training=False)
print("Transform :", transform)

In [ ]:
# ============================================================
# Cellule 4 — Choisir une image et calculer les probs par patch
# ============================================================
# ⚠️ Mets ici un filename présent dans ton dataset
FILENAME = "database1/img00024510.webp"

img_pil = Image.open(f"{IMG_DIR}/{FILENAME}").convert("RGB")
x = transform(img_pil).unsqueeze(0).to(DEVICE)

with torch.inference_mode():
    ratio, probs = model(x)            # probs : (1, 256, 3)

ratio = ratio.item()
probs = probs.squeeze(0).cpu().numpy()  # attendu (256, 3) avec GRID=16

print("probs.shape :", probs.shape, "  (attendu (256,3) avec GRID=16)")
assert probs.shape[0] == GRID * GRID, (
    f"GRID={GRID} -> {GRID*GRID} attendu, mais probs.shape[0]={probs.shape[0]}. "
    f"Ajuste GRID (et PATCH_PX = image_size // GRID) en conséquence."
)

# Remise en grille 16×16
p_bg  = probs[:, 0].reshape(GRID, GRID)   # décor
p_vis = probs[:, 1].reshape(GRID, GRID)   # visage visible
p_occ = probs[:, 2].reshape(GRID, GRID)   # visage occulté

print(f"Ratio d'occlusion estimé : {ratio:.4f}")

# Image de fond redimensionnée à 256×256 pour superposition
img_256 = img_pil.resize((256, 256))
img_np  = np.array(img_256) / 255.0       # (256, 256, 3) ∈ [0,1]

In [ ]:
# ============================================================
# Cellule 5 — Fonctions de superposition
# ============================================================
def upscale(grid_vals):
    """GRID×GRID -> 256×256 par répétition (chaque patch = PATCH_PX×PATCH_PX px)."""
    return np.kron(grid_vals, np.ones((PATCH_PX, PATCH_PX)))

def overlay_color(img, alpha_map, color):
    """
    Superpose une couleur unie sur l'image, avec opacité = alpha_map ∈ [0,1] par pixel.
    color : tuple RGB ∈ [0,1].
    """
    alpha = upscale(alpha_map)[..., None]          # (256,256,1)
    color = np.array(color).reshape(1, 1, 3)        # (1,1,3)
    return img * (1 - alpha) + color * alpha

def overlay_cmap(img, value_map, alpha_map, cmap_name="viridis"):
    """
    Superpose une cmap (value_map -> couleur), avec opacité = alpha_map.
    """
    cmap   = cm.get_cmap(cmap_name)
    colored = cmap(upscale(value_map))[..., :3]    # (256,256,3)
    alpha   = upscale(alpha_map)[..., None]
    return img * (1 - alpha) + colored * alpha

In [ ]:
# ============================================================
# Cellule 6 — Grille 10 images × 5 visualisations
# ============================================================
import pandas as pd

df = pd.read_csv("data/occlusion_datasets/train.csv").dropna()
sample = df.sample(10, random_state=None).reset_index(drop=True)

def compute_maps(filename):
    img_pil = Image.open(f"{IMG_DIR}/{filename}").convert("RGB")
    x = transform(img_pil).unsqueeze(0).to(DEVICE)
    with torch.inference_mode():
        ratio, probs = model(x)
    ratio = ratio.item()
    probs = probs.squeeze(0).cpu().numpy()
    p_bg  = probs[:, 0].reshape(GRID, GRID)
    p_vis = probs[:, 1].reshape(GRID, GRID)
    p_occ = probs[:, 2].reshape(GRID, GRID)
    img_np = np.array(img_pil.resize((256, 256))) / 255.0
    return img_np, p_bg, p_vis, p_occ, ratio

fig, axes = plt.subplots(10, 5, figsize=(20, 40))

col_titles = ["Originale", "Décor (bleu)", "Visage visible (vert)",
              "Occlusion (rouge)", "Ratio local (viridis)"]

for row in range(10):
    filename = sample.loc[row, "filename"]
    gt       = sample.loc[row, "FaceOcclusion"]
    img_np, p_bg, p_vis, p_occ, ratio = compute_maps(filename)

    axes[row, 0].imshow(img_np)
    axes[row, 1].imshow(overlay_color(img_np, p_bg,  color=(0, 0, 1)))
    axes[row, 2].imshow(overlay_color(img_np, p_vis, color=(0, 1, 0)))
    axes[row, 3].imshow(overlay_color(img_np, p_occ, color=(1, 0, 0)))

    ratio_local = p_occ / (p_vis + p_occ + 1e-8)
    alpha_face  = 1 - p_bg
    axes[row, 4].imshow(overlay_cmap(img_np, ratio_local, alpha_face, "viridis"))

    # GT vs estimation à gauche de chaque ligne
    axes[row, 0].set_ylabel(f"GT {gt:.3f}\nest. {ratio:.3f}", fontsize=12)

    for col in range(5):
        axes[row, col].set_xticks([])
        axes[row, col].set_yticks([])

for col, title in enumerate(col_titles):
    axes[0, col].set_title(title, fontsize=13)

plt.tight_layout()
plt.savefig("patch_visualization_grid_clean_trainval.png", dpi=100, bbox_inches="tight")
plt.show()